# Week 4 Final Project Code Notebook

This notebook brings together the Week 2 preparation checks and Week 3 exploration/model evaluation for the final report. Upload `crop_yield_prepared.csv` when prompted and run cells in order if you want to reproduce the analysis. The report is a separate DOCX deliverable.

**Important interpretation:** the target definition and units remain uncertain. The model is for learning and evaluation practice, not operational advice.


## 1. Load the prepared file

Upload the CSV you saved from Week 2. Keep your raw original unchanged. This notebook makes a separate in-memory analysis copy.


In [ ]:
import numpy as np
import pandas as pd
from google.colab import files

uploaded = files.upload()
csv_file = next(name for name in uploaded if name.lower().endswith(".csv"))
df = pd.read_csv(csv_file)

analysis_fields = [
    "Crop", "Season", "Crop_Year", "Annual_Rainfall",
    "Fertilizer", "Pesticide", "Yield"
]
missing_fields = [c for c in analysis_fields if c not in df.columns]
if missing_fields:
    raise ValueError(f"Required fields not found: {missing_fields}")

print("File:", csv_file)
print("Rows and columns:", df.shape)
print("Blank cells in analysis fields:", int(df[analysis_fields].isna().sum().sum()))
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Yield zeros:", int(df["Yield"].eq(0).sum()))
print("Negative Yield values:", int(df["Yield"].lt(0).sum()))


## Week 2 preparation validation

Confirm the prepared copy still has the expected shape and that the Week 2 text cleanup remains in place. This cell checks the file; it does not alter it.


In [ ]:
print("Rows and columns:", df.shape)
analysis_fields = ["Crop", "Season", "Crop_Year", "Annual_Rainfall", "Fertilizer", "Pesticide", "Yield"]
print("Blank cells in analysis fields:", int(df[analysis_fields].isna().sum().sum()))
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Crop labels with boundary spaces:", int(df["Crop"].astype("string").ne(df["Crop"].astype("string").str.strip()).sum()))
print("Season labels with boundary spaces:", int(df["Season"].astype("string").ne(df["Season"].astype("string").str.strip()).sum()))
print("Zero Yield rows:", int(df["Yield"].eq(0).sum()))
print("Rows by year:\n", df["Crop_Year"].value_counts().sort_index().to_string())


## 2. Explore target and coverage

Means are reported with medians because a few extreme values can pull the mean upward. Group summaries are descriptive only; they do not establish that a crop, season, or input caused a yield difference.


In [ ]:
print("Yield summary")
print(df["Yield"].describe(percentiles=[.25, .50, .75, .90, .95, .99]).round(3))
print("Yield skewness:", round(float(df["Yield"].skew()), 3))

print("\nRecords by year")
print(df["Crop_Year"].value_counts().sort_index().to_string())

print("\nYield by season")
print(df.groupby("Season")["Yield"].agg(["count", "median", "mean"]).round(3))

print("\nLargest crop groups")
crop_summary = (df.groupby("Crop")["Yield"].agg(["count", "median", "mean"])
                .sort_values("count", ascending=False).head(10).round(3))
print(crop_summary)

print("\nPooled Spearman correlation with Yield")
num_cols = ["Crop_Year", "Annual_Rainfall", "Fertilizer", "Pesticide", "Yield"]
print(df[num_cols].corr(method="spearman")["Yield"].sort_values().round(3))


## 3. Define the prediction question and time split

Target: the supplied `Yield` value. Predictors: crop, season, year, annual rainfall, fertilizer, and pesticide. Area and Production are intentionally excluded because they may directly determine the target.

The 2018 period is used for model selection. The 2019 period stays untouched until final evaluation. This better reflects prediction on a later year than a random row split.


In [ ]:
target = "Yield"
categorical = ["Crop", "Season"]
numeric = ["Crop_Year", "Annual_Rainfall", "Fertilizer", "Pesticide"]
features = categorical + numeric

work = df.dropna(subset=[target, "Crop_Year"]).copy()
train = work[work["Crop_Year"] <= 2017].copy()
valid = work[work["Crop_Year"] == 2018].copy()
test = work[work["Crop_Year"] == 2019].copy()
train_valid = work[work["Crop_Year"] <= 2018].copy()

print("Train through 2017:", len(train))
print("Validation 2018:", len(valid))
print("Test 2019:", len(test))
print("Excluded 2020:", int((work["Crop_Year"] == 2020).sum()))


## 4. Encode and transform without leakage

Crop and Season become one-hot indicators. Fertilizer and Pesticide use `log1p` because their distributions are strongly right-skewed. Numeric inputs are standardized. Category levels, means, and standard deviations are fitted from the training period and reused for validation and test data. The target also uses `log1p` during fitting and is converted back before scoring.


In [ ]:
def encode_fit(frame):
    vocab = {c: sorted(frame[c].astype(str).unique().tolist()) for c in categorical}
    cat_arrays = []
    for c in categorical:
        values = frame[c].astype(str).to_numpy()
        cat_arrays.append(np.column_stack([(values == level).astype(float) for level in vocab[c]]))

    nums = frame[numeric].astype(float).copy()
    for c in ["Fertilizer", "Pesticide"]:
        nums[c] = np.log1p(nums[c].clip(lower=0))
    means = nums.mean(axis=0).to_numpy()
    stds = nums.std(axis=0, ddof=0).replace(0, 1).to_numpy()
    nums_scaled = (nums.to_numpy() - means) / stds
    return np.column_stack([*cat_arrays, nums_scaled]), {"vocab": vocab, "means": means, "stds": stds}

def encode_apply(frame, fitted):
    cat_arrays = []
    for c in categorical:
        levels = fitted["vocab"][c]
        lookup = {level: i for i, level in enumerate(levels)}
        arr = np.zeros((len(frame), len(levels)), dtype=float)
        values = frame[c].astype(str).to_numpy()
        for row, value in enumerate(values):
            if value in lookup:
                arr[row, lookup[value]] = 1.0
        cat_arrays.append(arr)

    nums = frame[numeric].astype(float).copy()
    for c in ["Fertilizer", "Pesticide"]:
        nums[c] = np.log1p(nums[c].clip(lower=0))
    nums_scaled = (nums.to_numpy() - fitted["means"]) / fitted["stds"]
    return np.column_stack([*cat_arrays, nums_scaled])


## 5. Fit baseline and Ridge candidates

The baseline always predicts the training median. Ridge is fit to `log1p(Yield)` and uses an L2 penalty. The alpha setting is selected by the lowest 2018 validation MAE.


In [ ]:
def fit_ridge(X, y, alpha):
    X_aug = np.column_stack([np.ones(len(X)), X])
    penalty = np.eye(X_aug.shape[1]) * alpha
    penalty[0, 0] = 0  # do not penalize intercept
    return np.linalg.solve(X_aug.T @ X_aug + penalty, X_aug.T @ np.asarray(y, dtype=float))

def predict_ridge(X, coef):
    return np.column_stack([np.ones(len(X)), X]) @ coef

def score(y, pred):
    y = np.asarray(y, dtype=float)
    pred = np.asarray(pred, dtype=float)
    err = y - pred
    return {
        "MAE": float(np.mean(np.abs(err))),
        "RMSE": float(np.sqrt(np.mean(err ** 2))),
        "R2": float(1 - np.sum(err ** 2) / np.sum((y - y.mean()) ** 2)),
        "Median_AE": float(np.median(np.abs(err)))
    }

X_train, fit_train = encode_fit(train[features])
X_valid = encode_apply(valid[features], fit_train)
y_train = train[target].to_numpy()
y_valid = valid[target].to_numpy()

validation_results = {
    "median_baseline": score(y_valid, np.full(len(valid), np.median(y_train)))
}
for alpha in [0.1, 1.0, 10.0, 100.0]:
    coef = fit_ridge(X_train, np.log1p(y_train), alpha)
    pred = np.maximum(0, np.expm1(predict_ridge(X_valid, coef)))
    validation_results[f"ridge_alpha_{alpha:g}"] = score(y_valid, pred)

validation_table = pd.DataFrame(validation_results).T
print(validation_table.round(3))
selected = min(validation_results, key=lambda name: validation_results[name]["MAE"])
print("Selected on validation MAE:", selected)


## 6. Refit and evaluate on 2019

After selecting alpha on 2018, refit through 2018 and evaluate the 2019 holdout once. The 2020 records are not used for the main result.


In [ ]:
y_fit = train_valid[target].to_numpy()
y_test = test[target].to_numpy()
if selected == "median_baseline":
    test_pred = np.full(len(test), np.median(y_fit))
else:
    alpha = float(selected.replace("ridge_alpha_", ""))
    X_fit, fit_final = encode_fit(train_valid[features])
    X_test = encode_apply(test[features], fit_final)
    coef = fit_ridge(X_fit, np.log1p(y_fit), alpha)
    test_pred = np.maximum(0, np.expm1(predict_ridge(X_test, coef)))

final_metrics = score(y_test, test_pred)
baseline_pred = np.full(len(test), np.median(y_fit))
baseline_metrics = score(y_test, baseline_pred)
print("Ridge / selected model:", {k: round(v, 3) for k, v in final_metrics.items()})
print("Median baseline:", {k: round(v, 3) for k, v in baseline_metrics.items()})


## 7. Inspect tail and subgroup errors

A single overall score can hide poor performance on a crop group or extreme observations. This cell checks test records above the training 99th percentile and compares them with the rest. It also reports error by crop and season.


In [ ]:
predictions = test[["Crop", "Season", "Crop_Year", "Yield"]].copy()
predictions["Predicted_Yield"] = test_pred
predictions["Absolute_Error"] = np.abs(predictions["Yield"] - predictions["Predicted_Yield"])
train_p99 = float(train_valid["Yield"].quantile(.99))
within = predictions["Yield"] <= train_p99
print("Training 99th percentile:", round(train_p99, 3))
print("At/below threshold:", int(within.sum()), "Above:", int((~within).sum()))
print("Ridge MAE at/below threshold:", round(float(predictions.loc[within, "Absolute_Error"].mean()), 3))
print("Ridge absolute-error share above threshold:", round(float(predictions.loc[~within, "Absolute_Error"].sum() / predictions["Absolute_Error"].sum()), 3))
print("\nLargest test errors")
print(predictions.nlargest(10, "Absolute_Error").to_string(index=False))
print("\nError by crop (minimum 20 test rows)")
crop_error = predictions.groupby("Crop").agg(rows=("Yield", "size"), median_actual=("Yield", "median"), MAE=("Absolute_Error", "mean"))
print(crop_error[crop_error["rows"] >= 20].sort_values("MAE", ascending=False).head(15).round(3))
print("\nError by season")
season_error = predictions.groupby("Season").agg(rows=("Yield", "size"), median_actual=("Yield", "median"), MAE=("Absolute_Error", "mean"), median_AE=("Absolute_Error", "median"))
print(season_error.round(3))


## 8. Save evidence and write the interpretation

The notebook can save the test predictions for review. Before submitting, read the output tables and write your own interpretation. The model is not a causal analysis and is not ready to guide input use or procurement quantities. Confirm the Yield definition and units before presenting its estimates as real-world yield forecasts.


In [ ]:
predictions.to_csv("week3_test_predictions_2019.csv", index=False)
print("Saved week3_test_predictions_2019.csv")


## Final reporting notes

The analysis compares a training-median baseline with Ridge regression. It uses a chronological split, fits encoders and numeric transformations on training data only, and reports both overall and tail-sensitive errors.

Do not interpret pooled correlations as causal effects. Do not use the predictions for procurement volumes or farmer input advice until the target definition, units, source coverage, and multi-year performance are validated. The 2020 group is excluded from the primary test because its coverage is sparse.
